# ReFRAME library dataset analysis

In [1]:
import pandas as pd
from tqdm import tqdm
from collections import defaultdict

from benchmarking.metrics.base_metrics import (
    load_all_feature_tables,
)

# Load in data

In [2]:
ground_truth_library_data = pd.read_parquet(
    "../data/library_spectra/reframe_spikein_lib.pq"
)
ground_truth_ccs_data = pd.read_parquet(
    "../data/library_spectra/reframe_ms2s_with_ccs.parquet"
)

In [3]:
# Get the mzmine based feature tables only
annotation_types = [
    "annotated_spectral_entropy",
    "annotated_cosine_similarity",
    "annotated_dreams_similarity",
]

# Getting the relevant tables
combined_feature_table_list = defaultdict(pd.DataFrame)

for annotation_subfolder in annotation_types:
    combined_feature_table_list[annotation_subfolder] = load_all_feature_tables(
        ["../data/groundtruth_dataset/MSV000098263"],
        annotation_subfolder=annotation_subfolder,
    )

  0%|          | 0/1 [00:00<?, ?it/s]

Processing file: MSV000098263_metaboscape_annotated_spectral_entropy.parquet in dataset: ../data/groundtruth_dataset/MSV000098263
Processing file: MSV000098263_msdial_annotated_spectral_entropy.parquet in dataset: ../data/groundtruth_dataset/MSV000098263
Processing file: MSV000098263_mzmine_annotated_spectral_entropy.parquet in dataset: ../data/groundtruth_dataset/MSV000098263


  0%|          | 0/1 [00:00<?, ?it/s]

Processing file: MSV000098263_metaboscape_annotated_cosine_similarity.parquet in dataset: ../data/groundtruth_dataset/MSV000098263
Processing file: MSV000098263_msdial_annotated_cosine_similarity.parquet in dataset: ../data/groundtruth_dataset/MSV000098263
Processing file: MSV000098263_mzmine_annotated_cosine_similarity.parquet in dataset: ../data/groundtruth_dataset/MSV000098263


  0%|          | 0/1 [00:00<?, ?it/s]

Processing file: MSV000098263_msdial_annotated_dreams_similarity.parquet in dataset: ../data/groundtruth_dataset/MSV000098263
Processing file: MSV000098263_metaboscape_annotated_dreams_similarity.parquet in dataset: ../data/groundtruth_dataset/MSV000098263
Processing file: MSV000098263_mzmine_annotated_dreams_similarity.parquet in dataset: ../data/groundtruth_dataset/MSV000098263


100%|██████████| 1/1 [00:01<00:00,  1.67s/it]


In [4]:
common_annotations = {
    "annotated_spectral_entropy": defaultdict(int),
    "annotated_cosine_similarity": defaultdict(int),
    "annotated_dreams_similarity": defaultdict(int),
}

for ann_type, df_dict in tqdm(combined_feature_table_list.items()):
    for tool, df in df_dict.items():

        # Subset df to confident matches only
        subset_df = df[df["SCORE"] > 0.7]

        unique_annotations = subset_df["INCHIKEY"].unique()
        for inchi_key in unique_annotations:
            common_annotations[ann_type][inchi_key] += 1

100%|██████████| 3/3 [00:01<00:00,  1.59it/s]


In [5]:
# Restore annotations across all tools
common_across_tools = {
    ann_type: {inchi_key for inchi_key, count in ann_dict.items() if count == 3}
    for ann_type, ann_dict in common_annotations.items()
}

# Filter to annotations common across matching approaches
common_set = set.intersection(
    common_across_tools["annotated_spectral_entropy"],
    common_across_tools["annotated_cosine_similarity"],
    common_across_tools["annotated_dreams_similarity"],
)
len(common_set)

2310

In [ ]:
combined_feature_table_list["annotated_spectral_entropy"]["metaboscape"]